In [1]:
import os
import re
import numpy as np
import pandas as pd
from glob import glob
import math

In [2]:
BASE_DIR = "/users/6/mehta423/daycent/data/SAS_KGML_090925"
INPUT_DIR = os.path.join(BASE_DIR, "InputData")
OUTPUT_DIR = os.path.join(BASE_DIR, "OutputData_Synthetic_10000")
PROCESSED_DIR = "/users/6/mehta423/daycent/data/processed"

WEATHER_DIR = os.path.join(INPUT_DIR, "WeatherData")
INITC_FN = os.path.join(INPUT_DIR, "initial_site_conditions.xlsx")

MONTHLY_FN = os.path.join(OUTPUT_DIR, "SAS_scenario_1_monthly.csv")
HARVEST_FN = os.path.join(OUTPUT_DIR, "SAS_scenario_1_harvest.csv")

SCENARIOS_FN = os.path.join(INPUT_DIR, "schedule_scenarios_all_Synthetic_10000.csv")

In [3]:
all_points = []

for points in os.listdir(WEATHER_DIR):
    df = pd.read_csv(os.path.join(WEATHER_DIR, points))
    df['point_id'] = points.split(".csv")[0]
    all_points.append(df)

weather_df = pd.concat(all_points, ignore_index=True)
weather_df.head()

,Year,doy,Tmax,Tmin,Precip,point_id
0,2000,1,5.27,-4.83,0.000,1773513
1,2000,2,12.73,-1.85,0.000,1773513
2,2000,3,15.02,2.43,0.052,1773513
3,2000,4,10.48,0.62,1.026,1773513
4,2000,5,1.87,-5.29,0.041,1773513


In [4]:
# vocabulary of management events
MANAGEMENT_CLASSES = [
 'conventional_till_molboadplow','herbicide','soybean_planting',
 'nitrogen_fertilization_1.5gNm2','harvest_grain','cycle_end',
 'ryegrass_planting','nitrogen_fertilization_0gNm2',
 'reduced_till_tandemdisk','notill_rodweederrow','corn_planting',
 'nitrogen_fertilization_17.74gNm2','winterwheat_planting',
 'nitrogen_fertilization_10.312gNm2'
]

scenarios_df = pd.read_csv(SCENARIOS_FN)
scenarios_df = scenarios_df.rename({'simyear': 'Year'},  axis=1)

scenarios_df = scenarios_df.pivot_table(
    index=['scenario', 'Year', 'doy'], # Use all identifying columns for the index
    columns='management',
    aggfunc='size',
    fill_value=0
).reset_index()

# scenarios_df = scenarios_df[scenarios_df['scenario'] == 'scenario_1']
print(scenarios_df)

management       scenario  Year  doy  conventional_till_molboadplow  \
0              scenario_1  2000  132                              1   
1              scenario_1  2000  136                              0   
2              scenario_1  2000  137                              0   
3              scenario_1  2000  289                              0   
4              scenario_1  2000  294                              1   
...                   ...   ...  ...                            ...   
1086445     scenario_9999  2023  129                              0   
1086446     scenario_9999  2023  130                              0   
1086447     scenario_9999  2023  305                              0   
1086448     scenario_9999  2023  310                              0   
1086449     scenario_9999  2023  315                              0   

management  corn_planting  cycle_end  harvest_grain  herbicide  \
0                       0          0              0          0   
1              

In [7]:
X_daily = pd.merge(weather_df, scenarios_df, on=['Year', 'doy'], how='left')
X_daily.drop(columns=['scenario'], inplace=True)
X_daily.fillna(0, inplace=True)
X_daily

: 

: 

In [17]:
X_daily.isna().sum()

Year                                 0
doy                                  0
Tmax                                 0
Tmin                                 0
Precip                               0
point_id                             0
conventional_till_molboadplow        0
corn_planting                        0
cycle_end                            0
harvest_grain                        0
herbicide                            0
nitrogen_fertilization_0gNm2         0
nitrogen_fertilization_1.5gNm2       0
nitrogen_fertilization_10.312gNm2    0
nitrogen_fertilization_17.74gNm2     0
notill_rodweederrow                  0
reduced_till_tandemdisk              0
ryegrass_planting                    0
soybean_planting                     0
winterwheat_planting                 0
dtype: int64

In [7]:


files = os.listdir(OUTPUT_DIR)

pattern = re.compile(r"scenario_(\d+)")

numbers = sorted({(match.group(1)) for f in files if (match := pattern.search(f))})
print(numbers)

['1', '10', '100', '1000', '10000', '1001', '1002', '1003', '1004', '1005', '1006', '1007', '1008', '1009', '101', '1010', '1011', '1012', '1013', '1014', '1015', '1016', '1017', '1018', '1019', '102', '1020', '1021', '1022', '1023', '1024', '1025', '1026', '1027', '1028', '1029', '103', '1030', '1031', '1032', '1033', '1034', '1035', '1036', '1037', '1038', '1039', '104', '1040', '1041', '1042', '1043', '1044', '1045', '1046', '1047', '1048', '1049', '105', '1050', '1051', '1052', '1053', '1054', '1055', '1056', '1057', '1058', '1059', '106', '1060', '1061', '1062', '1063', '1064', '1065', '1066', '1067', '1068', '1069', '107', '1070', '1071', '1072', '1073', '1074', '1075', '1076', '1077', '1078', '1079', '108', '1080', '1081', '1082', '1083', '1084', '1085', '1086', '1087', '1088', '1089', '109', '1090', '1091', '1092', '1093', '1094', '1095', '1096', '1097', '1098', '1099', '11', '110', '1100', '1101', '1102', '1103', '1104', '1105', '1106', '1107', '1108', '1109', '111', '1110', '

In [8]:
def load_output_data(scenario_ids: list[str]):
    all_outputs = []
    month_to_doy = {1:30, 2:58, 3:89, 4:119, 5:150, 6:180, 7:211, 8:242, 9:272, 10:303, 11:333, 12:364}

    for scenario_id in scenario_ids:
        monthly_df = pd.read_csv(os.path.join(OUTPUT_DIR, f"SAS_scenario_{scenario_id}_monthly.csv"))
        monthly_df = monthly_df.rename({'id': 'point_id'}, axis=1)
        monthly_df['doy'] = monthly_df['month'].map(month_to_doy)
        monthly_df['simyear'] = monthly_df['simyear'].apply(lambda x: math.floor(float(x)))

        harvest_df = pd.read_csv(os.path.join(OUTPUT_DIR, f"SAS_scenario_{scenario_id}_harvest.csv"))
        harvest_df = harvest_df.rename({'id': 'point_id', 'dayofyr': 'doy'}, axis=1)

        output_df = pd.merge(monthly_df, harvest_df, on=['runid', 'point_id', 'simyear', 'doy'], how='outer')
        output_df = output_df.rename({'simyear': 'Year'}, axis=1)
        output_df['point_id'] = output_df['point_id'].astype(str)

        dates = pd.to_datetime(output_df['Year'].astype(str) + '-' + output_df['doy'].astype(str), format='%Y-%j')
        output_df['month'].fillna(dates.dt.month, inplace=True)
        output_df['month'] = output_df['month'].astype(int)

        output_df.sort_values(['point_id', 'Year', 'month', 'doy'], inplace=True)
        output_df.sort_index(inplace=True)

        output_df['scenario_id'] = scenario_id  # keep track of scenario
        all_outputs.append(output_df)

    return pd.concat(all_outputs, ignore_index=True)


def load_management_data(scenario_ids: list[str]):
    scenarios_df = pd.read_csv(SCENARIOS_FN).rename({'simyear': 'Year'}, axis=1)

    scenarios_df = scenarios_df.pivot_table(
        index=['scenario', 'Year', 'doy'],
        columns='management',
        aggfunc='size',
        fill_value=0
    ).reset_index()

    # filter for only requested scenarios
    scenarios_df = scenarios_df[scenarios_df['scenario'].isin([f'scenario_{sid}' for sid in scenario_ids])]
    scenarios_df['scenario_id'] = scenarios_df['scenario'].str.replace('scenario_', '', regex=False)

    return scenarios_df


def load_data(scenario_ids: list[str], weather_df: pd.DataFrame):
    print("Loading management data...")
    management_df = load_management_data(scenario_ids)

    X_daily = pd.merge(weather_df, management_df, on=['Year', 'doy'], how='left')
    X_daily.fillna(0, inplace=True)

    print("Loading output data...")
    Y = load_output_data(scenario_ids)

    return X_daily, Y


# Example usage:
X_daily, Y = load_data(numbers, weather_df)

print(X_daily.head())
print(Y.head())


Loading management data...
Loading output data...


/tmp/ipykernel_166034/752612133.py:19: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  output_df['month'].fillna(dates.dt.month, inplace=True)
/tmp/ipykernel_166034/752612133.py:19: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=T

FileNotFoundError: [Errno 2] No such file or directory: '/users/6/mehta423/daycent/data/SAS_KGML_090925/OutputData_Synthetic_10000/SAS_scenario_5398_monthly.csv'

In [ ]:
X_daily.to_csv(os.path.join(PROCESSED_DIR, "X_daily.csv"), index=False)
Y.to_csv(os.path.join(PROCESSED_DIR, "Y.csv"), index=False)

In [19]:
X_daily


,Year,doy,Tmax,Tmin,Precip,point_id,conventional_till_molboadplow,corn_planting,cycle_end,harvest_grain,herbicide,nitrogen_fertilization_0gNm2,nitrogen_fertilization_1.5gNm2,nitrogen_fertilization_10.312gNm2,nitrogen_fertilization_17.74gNm2,notill_rodweederrow,reduced_till_tandemdisk,ryegrass_planting,soybean_planting,winterwheat_planting
0,2000,1,5.270,-4.830,0.0000,1773513,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2000,2,12.730,-1.850,0.0000,1773513,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2000,3,15.020,2.430,0.0520,1773513,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2000,4,10.480,0.620,1.0260,1773513,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2000,5,1.870,-5.290,0.0410,1773513,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1853791,2024,362,12.818,-2.313,0.8766,710977,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1853792,2024,363,0.784,-5.354,0.0000,710977,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1853793,2024,364,4.451,-5.468,0.0062,710977,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1853794,2024,365,3.933,-3.492,0.1565,710977,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [20]:
Y

,runid,point_id,Year,month,somsc,doy,cgrain
0,1,657200,2000,10,NaN,289,170.656
1,1,657200,2001,1,5051.21,30,NaN
2,1,657200,2001,2,5050.51,58,NaN
3,1,657200,2001,3,5050.71,89,NaN
4,1,657200,2001,4,5051.53,119,NaN
...,...,...,...,...,...,...,...
59068,203,1799675,2023,10,NaN,289,175.356
59069,203,1799675,2023,10,5606.59,303,NaN
59070,203,1799675,2023,11,5606.73,333,NaN
59071,203,1799675,2023,12,5603.19,364,NaN


# Preprocessing

## Imputs processing

In [ ]:

df = X_daily

# Step 1: Drop leap day (doy == 366)
df = df[df["doy"] <= 365]
df = df.sort_values(by=["point_id", "Year", "doy"])

# Step 2: Select feature columns (include doy, exclude Year & point_id)
feature_cols = [c for c in df.columns if c not in ["Year", "point_id"]]

# Step 3: Group by point_id and Year
groups = df.groupby(["point_id", "Year"])

# Step 4: Create sequences and store mapping
sequences = []
mapping = []  # to store (point_id, year) for each sequence

for (pid, year), group in groups:
    if len(group) == 365:
        sequences.append(group[feature_cols].to_numpy())
        mapping.append((pid, year))  # store mapping info

# Convert to arrays
sequences = np.stack(sequences)  # shape: (num_sequences, 365, num_features)
mapping = np.array(mapping)      # shape: (num_sequences, 2)

# Step 5: Save both sequences and mapping
# Step 5: Save everything into one npy file
data_dict = {
    "data": sequences,
    "mapping": mapping,
    "columns": feature_cols
}

np.save(os.path.join(PROCESSED_DIR, "daycent_inputs.npy"), data_dict, allow_pickle=True)

In [22]:
feature_cols

['doy',
 'Tmax',
 'Tmin',
 'Precip',
 'conventional_till_molboadplow',
 'corn_planting',
 'cycle_end',
 'harvest_grain',
 'herbicide',
 'nitrogen_fertilization_0gNm2',
 'nitrogen_fertilization_1.5gNm2',
 'nitrogen_fertilization_10.312gNm2',
 'nitrogen_fertilization_17.74gNm2',
 'notill_rodweederrow',
 'reduced_till_tandemdisk',
 'ryegrass_planting',
 'soybean_planting',
 'winterwheat_planting']

In [13]:
sequences.shape

(5075, 365, 18)

In [15]:
# mapping = np.load("sequence_mapping.npy")
print(mapping[0])  # [point_id, Year] for first sequence


['1773513' '2000']


In [26]:
Y

,runid,point_id,Year,month,somsc,doy,cgrain
0,1,657200,2000,10,NaN,289,170.656
1,1,657200,2001,1,5051.21,30,NaN
2,1,657200,2001,2,5050.51,58,NaN
3,1,657200,2001,3,5050.71,89,NaN
4,1,657200,2001,4,5051.53,119,NaN
...,...,...,...,...,...,...,...
59068,203,1799675,2023,10,NaN,289,175.356
59069,203,1799675,2023,10,5606.59,303,NaN
59070,203,1799675,2023,11,5606.73,333,NaN
59071,203,1799675,2023,12,5603.19,364,NaN


In [25]:
for (pid, year), group in Y.groupby(["point_id", "Year"]):
    print(group)
    
    
    # Y_dict[(pid, year)] = group[["doy", "somsc", "cgrain"]].to_numpy()

       runid point_id  Year  month  somsc  doy   cgrain
29973    104  1773513  2000     10    NaN  289  162.738
       runid point_id  Year  month    somsc  doy   cgrain
29974    104  1773513  2001      1  4311.61   30      NaN
29975    104  1773513  2001      2  4310.76   58      NaN
29976    104  1773513  2001      3  4310.83   89      NaN
29977    104  1773513  2001      4  4311.45  119      NaN
29978    104  1773513  2001      5  4313.33  150      NaN
29979    104  1773513  2001      6  4274.77  180      NaN
29980    104  1773513  2001      7  4267.10  211      NaN
29981    104  1773513  2001      8  4266.48  242      NaN
29982    104  1773513  2001      9  4264.64  272      NaN
29983    104  1773513  2001     10      NaN  289  138.956
29984    104  1773513  2001     10  4261.90  303      NaN
29985    104  1773513  2001     11  4253.73  333      NaN
29986    104  1773513  2001     12  4239.40  364      NaN
       runid point_id  Year  month    somsc  doy   cgrain
29987    104  1773

In [29]:
# Ensure both point_id and Year are integers everywhere
df["point_id"] = df["point_id"].astype(int)
df["Year"] = df["Year"].astype(int)
Y["point_id"] = Y["point_id"].astype(int)
Y["Year"] = Y["Year"].astype(int)

# If mapping already exists as np.array of strings, convert it
mapping = np.array([[int(pid), int(year)] for pid, year in mapping])

# Build dictionary keyed by (point_id, Year)
Y_dict = {}
for (pid, year), group in Y.groupby(["point_id", "Year"]):
    Y_dict[(pid, year)] = group[["month", "doy", "somsc", "cgrain"]].to_numpy()

# Align Y to mapping
somsc_list = []
cgrain_list = []
for pid, year in mapping:
    if (pid, year) in Y_dict:
        data = Y_dict[(pid, year)]
        

        somsc_array = np.full(12, np.nan, dtype=np.float64)
        
        # The 'data' array has columns: 0=month, 1=doy, 2=somsc, 3=cgrain
        
        # 1a. Create a boolean mask to filter rows where 'somsc' (column index 2) is NOT NaN
        valid_somsc_mask = ~np.isnan(data[:, 2])
        
        # 1b. Filter the data to include only rows with valid somsc values
        valid_data = data[valid_somsc_mask]
        
        # 1c. Get the 0-indexed positions for assignment: month (column 0) - 1
        # We must ensure the indices are integers
        indices = valid_data[:, 0].astype(int) - 1
        
        # 1d. Get the corresponding somsc values (column 2)
        values = valid_data[:, 2]
        
        # 1e. Use advanced NumPy indexing for vectorized assignment
        # This is much faster than iterating row by row.
        # Note: If there are multiple somsc values for the same month, 
        # the last value encountered in the 'data' array (due to sorting in Y_dict) will be used.
        if indices.size > 0:
            somsc_array[indices] = values

            # 2. cgrain value: Must be a single number (no NaNs allowed in the source data)
        # Extract all cgrain values for this year (column index 3)
        cgrain_values = data[:, 3]
        
        # Filter out NaN values to find the single valid cgrain number
        valid_cgrain = cgrain_values[~np.isnan(cgrain_values)]
        
        # Append the results
        if valid_cgrain.size > 0:
            # Append the single annual cgrain value (the user guarantees it's unique/present)
            cgrain_list.append(valid_cgrain[0]) 
            
            # Append the 12-element monthly somsc array
            somsc_list.append(somsc_array)
        else:
            # Handle the case where Cgrain is unexpectedly missing (use NaN as a fallback)
            # print(f"Warning: cgrain value is missing for point_id {pid}, Year {year}. Appending NaN.")
            somsc_list.append(somsc_array)
            cgrain_list.append(np.nan)
        

    else:
        print(f"Missing data for point_id {pid}, Year {year}")
        somsc_list.append(np.empty((0, 2)))
        cgrain_list.append(np.empty((0, 2)))


# Convert lists to final NumPy arrays
final_somsc_array = np.array(somsc_list)
final_cgrain_array = np.array(cgrain_list)

# --- RESULTS ---
print("\n--- Final Results ---")
print("Mapping length:", len(mapping))
print("SOMSC List length:", len(somsc_list))
print("CGRAIN List length:", len(cgrain_list))

print(f"\nFinal SOMSC Array (Shape: {final_somsc_array.shape}):")
print(final_somsc_array)

print(f"\nFinal CGRAIN Array (Shape: {final_cgrain_array.shape}):")
print(final_cgrain_array)


--- Final Results ---
Mapping length: 5075
SOMSC List length: 5075
CGRAIN List length: 5075

Final SOMSC Array (Shape: (5075, 12)):
[[    nan     nan     nan ...     nan     nan     nan]
 [4311.61 4310.76 4310.83 ... 4261.9  4253.73 4239.4 ]
 [4234.75 4233.13 4232.55 ... 4191.9  4185.51 4172.3 ]
 ...
 [3876.55 3871.91 3868.79 ... 3875.65 3896.44 3883.99]
 [3876.74 3872.25 3869.08 ... 3792.01 3794.09 3791.78]
 [3791.75     nan     nan ...     nan     nan     nan]]

Final CGRAIN Array (Shape: (5075,)):
[162.738 138.956 106.714 ... 168.522 176.591     nan]


In [52]:
data_dict = {
    "somsc": final_somsc_array,
    "cgrain": final_cgrain_array,
}

np.save(os.path.join(PROCESSED_DIR, "daycent_outputs.npy"), data_dict, allow_pickle=True)

,runid,point_id,Year,month,somsc,doy,cgrain


In [63]:
idx = 0
Y[Y['point_id'] == '1773513'][Y['Year'] == mapping[idx,1]]

/tmp/ipykernel_27974/3854136065.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  Y[Y['point_id'] == '1773513'][Y['Year'] == mapping[idx,1]]


,runid,point_id,Year,month,somsc,doy,cgrain


In [61]:
Y

,runid,point_id,Year,month,somsc,doy,cgrain
0,1,657200,2000,10,NaN,289,170.656
1,1,657200,2001,1,5051.21,30,NaN
2,1,657200,2001,2,5050.51,58,NaN
3,1,657200,2001,3,5050.71,89,NaN
4,1,657200,2001,4,5051.53,119,NaN
...,...,...,...,...,...,...,...
59068,203,1799675,2023,10,NaN,289,175.356
59069,203,1799675,2023,10,5606.59,303,NaN
59070,203,1799675,2023,11,5606.73,333,NaN
59071,203,1799675,2023,12,5603.19,364,NaN
